**1 - Mount & Import**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import torch
import os
import pickle
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Device: cuda


**2 - Path & Load Data**

In [ ]:
path_d2      = "/content/drive/MyDrive/SKRIPSI/CODE/HASIL_SPLITTING"
path_lexicon = "/content/drive/MyDrive/SKRIPSI/CODE/LEXICON_V3"
path_prep    = "/content/drive/MyDrive/SKRIPSI/CODE/HASIL_PREPROCESSING"
save_path    = "/content/drive/MyDrive/SKRIPSI/CODE/HASIL MODELLING 12 SKEMA"
output_error = f"{save_path}/ERROR_ANALYSIS"
os.makedirs(output_error, exist_ok=True)

df2_test        = pd.read_csv(f"{path_d2}/data_test.csv")
df_lexicon_ib   = pd.read_csv(f"{path_lexicon}/lexicon_indobert.csv")
lexicon_ib_dict = dict(zip(df_lexicon_ib['kata'], df_lexicon_ib['skor_sarkasme']))

print(f"✅ Data test: {len(df2_test)} baris")
print(f"✅ Lexicon  : {len(lexicon_ib_dict)} kata")

✅ Data test: 508 baris
✅ Lexicon  : 2179 kata


**3 - Fungsi & Class Dataset (untuk reload IndoBERT Skema 3 & 9)**

In [ ]:
def hitung_skor_lexicon(teks, lexicon_dict):
    return sum(lexicon_dict.get(k, 0) for k in str(teks).split())

class SarkasmeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), max_length=self.max_len,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids'      : enc['input_ids'].squeeze(),
                'attention_mask' : enc['attention_mask'].squeeze(),
                'label'          : torch.tensor(self.labels[idx], dtype=torch.long)}

class SarkasmeDatasetLexicon(Dataset):
    def __init__(self, texts, labels, tokenizer, lexicon_dict, max_len=128):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.lexicon_dict, self.max_len = tokenizer, lexicon_dict, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc  = self.tokenizer(str(self.texts[idx]), max_length=self.max_len,
                              padding='max_length', truncation=True, return_tensors='pt')
        skor = hitung_skor_lexicon(self.texts[idx], self.lexicon_dict)
        return {'input_ids'      : enc['input_ids'].squeeze(),
                'attention_mask' : enc['attention_mask'].squeeze(),
                'lexicon_score'  : torch.tensor(skor, dtype=torch.float),
                'label'          : torch.tensor(self.labels[idx], dtype=torch.long)}

print("✅ Class dataset siap!")

✅ Class dataset siap!


**4 - Class Model Skema 9 (IndoBERT + Lexicon)**

In [ ]:
class IndoBERTWithLexicon(torch.nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()
        self.bert       = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
        self.classifier = torch.nn.Linear(num_labels + 1, num_labels)

    def forward(self, input_ids, attention_mask, lexicon_score, labels=None):
        logits       = self.bert(input_ids=input_ids, attention_mask=attention_mask).logits
        combined     = torch.cat([logits, lexicon_score.unsqueeze(1)], dim=1)
        final_logits = self.classifier(combined)
        loss = torch.nn.CrossEntropyLoss()(final_logits, labels) if labels is not None else None
        return loss, final_logits

print("✅ Class IndoBERTWithLexicon siap!")

✅ Class IndoBERTWithLexicon siap!


**5 - Load & Prediksi Skema 3 (IndoBERT D2, Tanpa Lexicon)**

In [ ]:
print("🔍 Load Skema 3...")
path_s3  = f"{save_path}/Skema_3_IndoBERT_D2_-_Tanpa_Lexicon"
tok_s3   = AutoTokenizer.from_pretrained(path_s3)
model_s3 = AutoModelForSequenceClassification.from_pretrained(path_s3).to(device)
model_s3.eval()

loader_s3 = DataLoader(
    SarkasmeDataset(df2_test['text_clean'].tolist(), df2_test['label'].tolist(), tok_s3),
    batch_size=16
)

preds_s3, labels_s3 = [], []
with torch.no_grad():
    for batch in loader_s3:
        out = model_s3(input_ids=batch['input_ids'].to(device),
                       attention_mask=batch['attention_mask'].to(device))
        preds_s3.extend(torch.argmax(out.logits, dim=1).cpu().numpy())
        labels_s3.extend(batch['label'].numpy())

print("✅ Prediksi Skema 3 selesai!")
del model_s3; torch.cuda.empty_cache()

🔍 Load Skema 3...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Prediksi Skema 3 selesai!


**6 - Load & Prediksi Skema 9 (IndoBERT D2 + Lexicon)**

In [ ]:
print("🔍 Load Skema 9...")
path_s9  = f"{save_path}/Skema_9_IndoBERT_D2+Lexicon"
tok_s9   = AutoTokenizer.from_pretrained(path_s9)
model_s9 = IndoBERTWithLexicon("indobenchmark/indobert-base-p1").to(device)
model_s9.load_state_dict(torch.load(f"{path_s9}/model.pt", map_location=device))
model_s9.eval()

loader_s9 = DataLoader(
    SarkasmeDatasetLexicon(df2_test['text_clean'].tolist(), df2_test['label'].tolist(), tok_s9, lexicon_ib_dict),
    batch_size=16
)

preds_s9, labels_s9 = [], []
with torch.no_grad():
    for batch in loader_s9:
        _, logits = model_s9(input_ids=batch['input_ids'].to(device),
                             attention_mask=batch['attention_mask'].to(device),
                             lexicon_score=batch['lexicon_score'].to(device))
        preds_s9.extend(torch.argmax(logits, dim=1).cpu().numpy())
        labels_s9.extend(batch['label'].numpy())

print("✅ Prediksi Skema 9 selesai!")
del model_s9; torch.cuda.empty_cache()

🔍 Load Skema 9...


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `5`.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Prediksi Skema 9 selesai!


**7 - Load Prediksi Skema 4 & 10 (SVM) dari all_preds.pkl**

In [ ]:
pred_path = f"{save_path}/all_preds.pkl"
if not os.path.exists(pred_path):
    raise FileNotFoundError("all_preds.pkl tidak ditemukan — jalankan dulu notebook modeling.")

with open(pred_path, "rb") as f:
    all_preds = pickle.load(f)

print("✅ Skema yang tersedia di all_preds.pkl:")
for k in all_preds:
    print(f"   - {k}")

nama_s4  = 'Skema 4 SVM D2 - Tanpa Lexicon'
nama_s10 = 'Skema 10 SVM D2+Lexicon'

y_true_s4,  preds_s4,  prob_s4  = all_preds[nama_s4]
y_true_s10, preds_s10, prob_s10 = all_preds[nama_s10]

print(f"\n✅ Skema 4  : {len(preds_s4)} prediksi dimuat")
print(f"✅ Skema 10 : {len(preds_s10)} prediksi dimuat")

✅ Skema yang tersedia di all_preds.pkl:
   - Skema 1 IndoBERT D1 - Tanpa Lexicon
   - Skema 2 SVM D1 - Tanpa Lexicon
   - Skema 3 IndoBERT D2 - Tanpa Lexicon
   - Skema 4 SVM D2 - Tanpa Lexicon
   - Skema 5 IndoBERT D1+D2 - Tanpa Lexicon
   - Skema 6 SVM D1+D2 - Tanpa Lexicon
   - Skema 7 IndoBERT D1+Lexicon
   - Skema 8 SVM D1+Lexicon
   - Skema 9 IndoBERT D2+Lexicon
   - Skema 10 SVM D2+Lexicon
   - Skema 11 IndoBERT D1+D2+Lexicon
   - Skema 12 SVM D1+D2+Lexicon

✅ Skema 4  : 508 prediksi dimuat
✅ Skema 10 : 508 prediksi dimuat


**8 - Cross-check & Verifikasi Urutan Data**

In [ ]:
print("🔍 Cross-check metrik (bandingkan dengan Tabel 4.21/4.22):")
for nama, (yt, yp, _) in [(nama_s4, (y_true_s4, preds_s4, None)),
                          (nama_s10, (y_true_s10, preds_s10, None))]:
    acc = accuracy_score(yt, yp)
    f1  = f1_score(yt, yp, average='weighted')
    print(f"   {nama:35s} → Acc: {acc:.4f} | F1: {f1:.4f}")

print("\n🔍 Verifikasi urutan data (y_true pkl vs df2_test['label']):")
for nama, yt in [(nama_s4, y_true_s4), (nama_s10, y_true_s10)]:
    cocok  = np.array_equal(np.array(yt), df2_test['label'].values)
    status = "✅ COCOK" if cocok else "❌ TIDAK COCOK - urutan data beda, JANGAN lanjut!"
    print(f"   {nama:35s} → {status}")

🔍 Cross-check metrik (bandingkan dengan Tabel 4.21/4.22):
   Skema 4 SVM D2 - Tanpa Lexicon      → Acc: 0.9055 | F1: 0.9055
   Skema 10 SVM D2+Lexicon             → Acc: 0.8898 | F1: 0.8834

🔍 Verifikasi urutan data (y_true pkl vs df2_test['label']):
   Skema 4 SVM D2 - Tanpa Lexicon      → ✅ COCOK
   Skema 10 SVM D2+Lexicon             → ✅ COCOK


**9 - Pastikan full_text Tersedia**

In [ ]:
if 'full_text' not in df2_test.columns:
    df_p1 = pd.read_csv(f"{path_prep}/data_anotasi_fase1_pra_implementasi.csv")
    df_p2 = pd.read_csv(f"{path_prep}/data_anotasi_fase2_implementasi.csv")
    df_prep = pd.concat([df_p1, df_p2], ignore_index=True)
    mapping = df_prep[['text_clean', 'full_text']].drop_duplicates(subset='text_clean')
    df2_test = df2_test.merge(mapping, on='text_clean', how='left')
    print(f"✅ full_text digabung. Match: {df2_test['full_text'].notna().sum()} dari {len(df2_test)}")
else:
    print("✅ full_text sudah tersedia di data_test.csv")

✅ full_text digabung. Match: 508 dari 508


**10 - Fungsi Bantu**

In [ ]:
def ekstrak_error(df_test, y_true, y_pred, nama_skema):
    df_hasil = df_test.copy().reset_index(drop=True)
    df_hasil['label_asli']     = np.array(y_true)
    df_hasil['label_prediksi'] = np.array(y_pred)

    kolom = ['full_text', 'text_clean', 'label_asli', 'label_prediksi']
    df_fn = df_hasil[(df_hasil['label_asli'] == 1) & (df_hasil['label_prediksi'] == 0)][kolom].reset_index(drop=True)
    df_fp = df_hasil[(df_hasil['label_asli'] == 0) & (df_hasil['label_prediksi'] == 1)][kolom].reset_index(drop=True)
    df_fn.index += 1
    df_fp.index += 1

    print(f"{nama_skema} - FN: {len(df_fn)} | FP: {len(df_fp)}")
    return df_fn, df_fp

def ambil_sample(df, n=15, seed=42):
    if len(df) <= n:
        return df
    return df.sample(n=n, random_state=seed).sort_index()

print("✅ Fungsi bantu siap!")

✅ Fungsi bantu siap!


**11 - Ekstrak Error Analysis Seluruh Skema**

In [ ]:
df_fn_s3,  df_fp_s3  = ekstrak_error(df2_test, labels_s3,  preds_s3,  "Skema 3 (IndoBERT D2)")
df_fn_s9,  df_fp_s9  = ekstrak_error(df2_test, labels_s9,  preds_s9,  "Skema 9 (IndoBERT D2+Lexicon)")
df_fn_s4,  df_fp_s4  = ekstrak_error(df2_test, y_true_s4,  preds_s4,  "Skema 4 (SVM D2)")
df_fn_s10, df_fp_s10 = ekstrak_error(df2_test, y_true_s10, preds_s10, "Skema 10 (SVM D2+Lexicon)")

Skema 3 (IndoBERT D2) - FN: 22 | FP: 33
Skema 9 (IndoBERT D2+Lexicon) - FN: 49 | FP: 32
Skema 4 (SVM D2) - FN: 24 | FP: 24
Skema 10 (SVM D2+Lexicon) - FN: 37 | FP: 19


**12 - Sample 15 Data per Kategori (Lampiran 2)**

In [ ]:
sample_fn_s3  = ambil_sample(df_fn_s3)
sample_fp_s3  = ambil_sample(df_fp_s3)
sample_fn_s9  = ambil_sample(df_fn_s9)
sample_fp_s9  = ambil_sample(df_fp_s9)
sample_fn_s4  = ambil_sample(df_fn_s4)
sample_fp_s4  = ambil_sample(df_fp_s4)
sample_fn_s10 = ambil_sample(df_fn_s10)
sample_fp_s10 = ambil_sample(df_fp_s10)

print("✅ Sample untuk Lampiran 2 siap!")

✅ Sample untuk Lampiran 2 siap!


**13 - Simpan ke Excel**

In [ ]:
with pd.ExcelWriter(f'{output_error}/error_analysis_S3_S4_S9_S10.xlsx') as writer:
    df_fn_s3.to_excel(writer,  sheet_name='S3_FN_Full',  index=True)
    df_fp_s3.to_excel(writer,  sheet_name='S3_FP_Full',  index=True)
    df_fn_s4.to_excel(writer,  sheet_name='S4_FN_Full',  index=True)
    df_fp_s4.to_excel(writer,  sheet_name='S4_FP_Full',  index=True)
    df_fn_s9.to_excel(writer,  sheet_name='S9_FN_Full',  index=True)
    df_fp_s9.to_excel(writer,  sheet_name='S9_FP_Full',  index=True)
    df_fn_s10.to_excel(writer, sheet_name='S10_FN_Full', index=True)
    df_fp_s10.to_excel(writer, sheet_name='S10_FP_Full', index=True)

    sample_fn_s3.to_excel(writer,  sheet_name='S3_FN_Sample15',  index=True)
    sample_fp_s3.to_excel(writer,  sheet_name='S3_FP_Sample15',  index=True)
    sample_fn_s4.to_excel(writer,  sheet_name='S4_FN_Sample15',  index=True)
    sample_fp_s4.to_excel(writer,  sheet_name='S4_FP_Sample15',  index=True)
    sample_fn_s9.to_excel(writer,  sheet_name='S9_FN_Sample15',  index=True)
    sample_fp_s9.to_excel(writer,  sheet_name='S9_FP_Sample15',  index=True)
    sample_fn_s10.to_excel(writer, sheet_name='S10_FN_Sample15', index=True)
    sample_fp_s10.to_excel(writer, sheet_name='S10_FP_Sample15', index=True)

print("✅ Tersimpan: error_analysis_S3_S4_S9_S10.xlsx (16 sheet)")

✅ Tersimpan: error_analysis_S3_S4_S9_S10.xlsx (16 sheet)


**14 - Gabung FN + FP per Skema (1 Tabel, Kolom "Jenis Kesalahan")**

In [ ]:
def gabung_fn_fp(df_fn, df_fp):
    df_fn = df_fn.copy()
    df_fp = df_fp.copy()
    df_fn['Jenis Kesalahan'] = 'False Negative'
    df_fp['Jenis Kesalahan'] = 'False Positive'

    df_gabung = pd.concat([df_fn, df_fp], ignore_index=True)

    label_map = {0: 'Non-Sarkasme', 1: 'Sarkasme'}
    df_gabung['label_asli']     = df_gabung['label_asli'].map(label_map)
    df_gabung['label_prediksi'] = df_gabung['label_prediksi'].map(label_map)

    df_gabung = df_gabung.rename(columns={
        'full_text'     : 'Tweet',
        'label_asli'    : 'Label Asli',
        'label_prediksi': 'Label Prediksi'
    })[['Tweet', 'Label Asli', 'Label Prediksi', 'Jenis Kesalahan']]

    df_gabung.index = range(1, len(df_gabung) + 1)
    df_gabung.index.name = 'No'
    return df_gabung

lampiran_s3  = gabung_fn_fp(sample_fn_s3,  sample_fp_s3)
lampiran_s4  = gabung_fn_fp(sample_fn_s4,  sample_fp_s4)
lampiran_s9  = gabung_fn_fp(sample_fn_s9,  sample_fp_s9)
lampiran_s10 = gabung_fn_fp(sample_fn_s10, sample_fp_s10)

print(f"Skema 3  : {len(lampiran_s3)} baris")
print(f"Skema 4  : {len(lampiran_s4)} baris")
print(f"Skema 9  : {len(lampiran_s9)} baris")
print(f"Skema 10 : {len(lampiran_s10)} baris")

lampiran_s3.head()

Skema 3  : 30 baris
Skema 4  : 30 baris
Skema 9  : 30 baris
Skema 10 : 30 baris


,Tweet,Label Asli,Label Prediksi,Jenis Kesalahan
No,,,,
1,halo guys welkam bek tu my cenel... hari ini k...,Sarkasme,Non-Sarkasme,False Negative
2,Kasus pengoplosan ialah bentuk pengalihan isu ...,Sarkasme,Non-Sarkasme,False Negative
3,Pemerintah menganggarkan Rp 10.000 untuk progr...,Sarkasme,Non-Sarkasme,False Negative
4,Kondangan adalah makan bergizi gratis tapi tid...,Sarkasme,Non-Sarkasme,False Negative
5,Kemudian.. Kemudian.. Kita pangkas anggaran la...,Sarkasme,Non-Sarkasme,False Negative


**15 - Simpan ke Excel**

In [ ]:
with pd.ExcelWriter(f'{output_error}/lampiran2_error_analysis.xlsx') as writer:
    lampiran_s3.to_excel(writer,  sheet_name='Lampiran_Skema3')
    lampiran_s4.to_excel(writer,  sheet_name='Lampiran_Skema4')
    lampiran_s9.to_excel(writer,  sheet_name='Lampiran_Skema9')
    lampiran_s10.to_excel(writer, sheet_name='Lampiran_Skema10')

print("✅ Tersimpan: lampiran2_error_analysis.xlsx (4 sheet, 1 tabel per skema)")
print(f"   Lokasi: {output_error}")

✅ Tersimpan: lampiran2_error_analysis.xlsx (4 sheet, 1 tabel per skema)
   Lokasi: /content/drive/MyDrive/SKRIPSI/CODE/HASIL MODELLING 12 SKEMA/ERROR_ANALYSIS
